# NLP Preprocessing Engine 
In this notebook, I built a robust NLP preprocessing pipeline for messy real-world text.


## Task 1: Conceptual Understanding

### 1. Difference between "Love" and "love"
In NLP, "Love" and "love" are treated as different words because text is case-sensitive by default. This can create duplicate representations of the same word. Converting everything to lowercase helps maintain consistency.

### 2. What happens if stopwords are not removed?
If stopwords like "is", "the", "and" are not removed, they increase noise in the dataset and do not contribute much to the meaning. This can reduce model efficiency and accuracy.

### 3. When removing stopwords can be harmful?
- In sentiment analysis, words like "not" are important (e.g., "not good").
- In question-answering systems, words like "what", "where", "who" are necessary for understanding context.

### 4. Stemming vs Lemmatization
- Stemming cuts words to a root form (e.g., "playing" → "play"), sometimes incorrectly.
- Lemmatization converts words into meaningful base forms (e.g., "better" → "good"), making it more accurate.

In [8]:
import re
from collections import Counter


## Task 2: Building the Preprocessing Function

I designed a function that performs multiple cleaning steps:

- Convert text to lowercase
- Remove URLs and email-like patterns
- Remove numbers
- Handle repeated characters
- Remove unwanted symbols/emojis
- Remove very short words (except "no" and "not")
- Clean extra spaces

The function returns:
1. Clean tokens
2. Clean sentence

## Preprocessing Function


In [9]:
EXCEPTION_WORDS = {"no", "not"}

def preprocess_text(text):
    
    
    if not isinstance(text, str) or text.strip() == "":
        return [], ""
    
    
    text = text.lower()
    
 
    text = re.sub(r"http\S+|www\S+|@\S+", "", text)
    
    #
    text = re.sub(r"\d+", "", text)
    
    
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    
  
    text = re.sub(r"[^\w\s]", "", text)
    
  
    text = re.sub(r"\s+", " ", text).strip()
    
    
    tokens = text.split()
    
   
    cleaned_tokens = []
    for word in tokens:
        if len(word) > 2 or word in EXCEPTION_WORDS:
            cleaned_tokens.append(word)
    
    cleaned_sentence = " ".join(cleaned_tokens)
    
    return cleaned_tokens, cleaned_sentence

## Task 3: Stress Testing

Now I will test the function on different types of real-world noisy text inputs including:
- Emojis
- Numbers
- URLs
- Repeated characters
- Mixed cases

In [10]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

In [11]:
for text in test_sentences:
    tokens, clean_sent = preprocess_text(text)
    
    print("Original Text :", text)
    print("Tokens        :", tokens)
    print("Cleaned Text  :", clean_sent)
    print("-"*50)
    
    results.append((tokens, clean_sent))

Original Text : Get 100% FREE access now!!!
Tokens        : ['get', 'free', 'access', 'now']
Cleaned Text  : get free access now
--------------------------------------------------
Original Text : I absolutely looooved this product 😍😍
Tokens        : ['absolutely', 'loved', 'this', 'product']
Cleaned Text  : absolutely loved this product
--------------------------------------------------
Original Text : Worst service ever... 0/10
Tokens        : ['worst', 'service', 'ever']
Cleaned Text  : worst service ever
--------------------------------------------------
Original Text : Call me at 9876543210
Tokens        : ['call']
Cleaned Text  : call
--------------------------------------------------
Original Text : This is THE best course!!!
Tokens        : ['this', 'the', 'best', 'course']
Cleaned Text  : this the best course
--------------------------------------------------
Original Text : Visit https://openai.com now!
Tokens        : ['visit', 'now']
Cleaned Text  : visit now
---------------

## Task 4: Token Analytics

For each processed sentence, I calculated:
- Total tokens
- Unique tokens
- Average token length

In [12]:
for i, (tokens, sent) in enumerate(results):
    
    if len(tokens) == 0:
        continue
    
    total = len(tokens)
    unique = len(set(tokens))
    avg_len = sum(len(word) for word in tokens) / total
    
    print(f"Sentence {i+1}")
    print("Total Tokens     :", total)
    print("Unique Tokens    :", unique)
    print("Avg Token Length :", round(avg_len, 2))
    print("-"*40)

Sentence 1
Total Tokens     : 4
Unique Tokens    : 4
Avg Token Length : 4.0
----------------------------------------
Sentence 2
Total Tokens     : 4
Unique Tokens    : 4
Avg Token Length : 6.5
----------------------------------------
Sentence 3
Total Tokens     : 3
Unique Tokens    : 3
Avg Token Length : 5.33
----------------------------------------
Sentence 4
Total Tokens     : 1
Unique Tokens    : 1
Avg Token Length : 4.0
----------------------------------------
Sentence 5
Total Tokens     : 4
Unique Tokens    : 4
Avg Token Length : 4.25
----------------------------------------
Sentence 6
Total Tokens     : 2
Unique Tokens    : 2
Avg Token Length : 4.0
----------------------------------------
Sentence 7
Total Tokens     : 3
Unique Tokens    : 3
Avg Token Length : 3.0
----------------------------------------
Sentence 8
Total Tokens     : 1
Unique Tokens    : 1
Avg Token Length : 3.0
----------------------------------------
Sentence 9
Total Tokens     : 4
Unique Tokens    : 4
Avg Token

### Observations

- The sentence with the most noise was:
  "Get 100% FREE access now!!!" because it had numbers, symbols, and uppercase words.

- The sentence that retained the most meaningful tokens:
  "I am not happy with this" because it preserved important sentiment words like "not".

## Task 5: Frequency Analysis

Here I combined all tokens and found:
- Most frequent words
- Least frequent words

In [13]:
all_tokens = []

for tokens, _ in results:
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10 Frequent Words:")
print(counter.most_common(10))

print("\nLeast 5 Frequent Words:")
print(counter.most_common()[-5:])

Top 10 Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Least 5 Frequent Words:
[('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


## Task 6: Full Pipeline

This function processes a list of sentences and returns:
- All tokens
- Cleaned sentences

In [14]:
def full_pipeline(text_list):
    
    all_tokens = []
    clean_sentences = []
    
    for text in text_list:
        tokens, clean_sent = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean_sent)
    
    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

output = full_pipeline(test_sentences)
print(output)

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely loved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited offer', 'not happy with this']}


## Task 7: Error Handling

Testing edge cases like:
- Empty input
- Only emojis
- Only numbers

In [15]:
edge_cases = ["", "😂😂😂😂", "1234567890"]

for case in edge_cases:
    tokens, sent = preprocess_text(case)
    print("Input   :", case)
    print("Tokens  :", tokens)
    print("Output  :", sent)
    print("-"*40)

Input   : 
Tokens  : []
Output  : 
----------------------------------------
Input   : 😂😂😂😂
Tokens  : []
Output  : 
----------------------------------------
Input   : 1234567890
Tokens  : []
Output  : 
----------------------------------------


## Conclusion

This preprocessing engine successfully handles messy real-world text data by applying multiple cleaning techniques.

The modular structure makes it reusable and scalable for real NLP applications such as sentiment analysis, chatbots, and recommendation systems.